### Sales&Profit Performance MV

In [0]:

CREATE OR REFRESH MATERIALIZED VIEW mavenmarket.gold.gold_sales_profit_performance
COMMENT "Materialized View: Gold layer aggregated monthly sales performance with time intelligence metrics"
AS
WITH base_monthly_sales AS (
    SELECT 
        -- Monthly Date Groupings (from silver_calender)
        d.year AS year_number,
        d.quarter AS quarter_number,
        d.month AS month_number,
        d.month_name,
        d.year_month AS year_month_code,

        -- Store Attributes (from silver_stores)
        s.store_id,
        s.store_name,
        s.store_type AS store_type_description,
        s.store_city,
        s.store_state,
        s.store_country,

        -- Region Attributes (Joined via region_id from silver_regions)
        r.region_id,
        r.sales_region AS sales_region_name,
        r.sales_district AS sales_district_name,

        -- Product Attributes (from silver_products)
        p.product_id,
        p.product_name,
        p.product_brand AS brand_name,
        p.product_category AS category_name,
        p.product_sku AS sku_code,

        -- Transaction Channel (from silver_transactions)
        t.transaction_mode,

        -- Monthly Aggregated Financial Measures
        COUNT(DISTINCT t.customer_id) AS total_orders_count,
        SUM(CAST(t.quantity AS INT)) AS total_units_sold,
        SUM(CAST(t.quantity AS INT) * p.product_retail_price) AS gross_revenue_amount,
        SUM(CAST(t.quantity AS INT) * p.product_cost) AS total_cost_amount,
        SUM(CAST(t.quantity AS INT) * (p.product_retail_price - p.product_cost)) AS gross_profit_amount

    FROM mavenmarket.silver.silver_transactions t
    JOIN mavenmarket.silver.silver_calendar d 
        ON CAST(t.transaction_date AS DATE) = d.calendar_date
    JOIN mavenmarket.silver.silver_stores s 
        ON CAST(t.store_id AS INT) = s.store_id
    JOIN mavenmarket.silver.silver_regions r 
        ON CAST(s.region_id AS INT) = CAST(r.region_id AS INT)
    JOIN mavenmarket.silver.silver_products p 
        ON CAST(t.product_id AS INT) = p.product_id

    GROUP BY ALL
),
sales_with_time_intelligence AS (
    SELECT 
        *,
        -- Profit Margin Percentage
        ROUND(
            (gross_profit_amount / NULLIF(gross_revenue_amount, 0)) * 100.0, 
            2
        ) AS profit_margin_pct,

        -- Average Order Value (AOV)
        ROUND(
            gross_revenue_amount / NULLIF(total_orders_count, 0), 
            2
        ) AS average_order_value,

        -- Prior Year Monthly Revenue (Looks back 12 true monthly partition buckets)
        LAG(gross_revenue_amount, 12) OVER (
            PARTITION BY store_id, product_id, transaction_mode 
            ORDER BY year_number, month_number
        ) AS prior_year_gross_revenue,

        -- Prior Year Monthly Profit
        LAG(gross_profit_amount, 12) OVER (
            PARTITION BY store_id, product_id, transaction_mode 
            ORDER BY year_number, month_number
        ) AS prior_year_gross_profit

    FROM base_monthly_sales
)
SELECT 
    *,
    -- Revenue YoY Growth Calculations
    CASE 
        WHEN prior_year_gross_revenue IS NULL THEN NULL
        ELSE ROUND(gross_revenue_amount - prior_year_gross_revenue, 2)
    END AS revenue_yoy_growth_amount,
    
    ROUND(
        ((gross_revenue_amount - prior_year_gross_revenue) / NULLIF(prior_year_gross_revenue, 0)) * 100.0, 
        2
    ) AS revenue_yoy_growth_pct,

    -- Profit YoY Growth Calculations
    CASE 
        WHEN prior_year_gross_profit IS NULL THEN NULL
        ELSE ROUND(gross_profit_amount - prior_year_gross_profit, 2)
    END AS profit_yoy_growth_amount,

    ROUND(
        ((gross_profit_amount - prior_year_gross_profit) / NULLIF(prior_year_gross_profit, 0)) * 100.0, 
        2
    ) AS profit_yoy_growth_pct

FROM sales_with_time_intelligence;

### Customer Analytics Performance MV

In [0]:
CREATE OR REFRESH MATERIALIZED VIEW mavenmarket.gold.gold_customer_analytics_performance
COMMENT "Materialized View: Gold layer aggregated customer performance dynamic across all silver_customers entries"
AS
WITH clean_customers AS (
    -- Dynamically cast every single record in silver_customers 
    SELECT 
        CAST(FLOOR(CAST(c.customer_id AS DOUBLE)) AS INT) AS customer_id,
        CONCAT(COALESCE(c.first_name, ''), ' ', COALESCE(c.last_name, '')) AS customer_name,
        c.customer_city,
        c.customer_state_province AS customer_state,
        c.customer_country,
        c.gender,
        c.marital_status,
        c.yearly_income AS yearly_income_amount,
        c.num_children_at_home AS children_at_home_count,
        c.total_children AS total_children_count,
        c.education AS education_level,
        c.occupation AS occupation_name,
        c.homeowner AS is_homeowner_flag,
        ROW_NUMBER() OVER (
            PARTITION BY CAST(FLOOR(CAST(c.customer_id AS DOUBLE)) AS INT) 
            ORDER BY c.customer_id ASC
        ) AS dedup_rank
    FROM mavenmarket.silver.silver_customers c
    WHERE c.customer_id IS NOT NULL
),
master_customers AS (
    -- Guarantees the exact count matching silver_customers (60,000)
    SELECT * 
    FROM clean_customers 
    WHERE dedup_rank = 1
),
clean_transactions AS (
    -- Safely parse all transaction history
    SELECT 
        CAST(FLOOR(CAST(t.customer_id AS DOUBLE)) AS INT) AS customer_id,
        CAST(FLOOR(CAST(t.store_id AS DOUBLE)) AS INT) AS store_id,
        CAST(FLOOR(CAST(t.product_id AS DOUBLE)) AS INT) AS product_id,
        CAST(t.transaction_date AS DATE) AS transaction_date,
        CAST(t.quantity AS INT) AS quantity
    FROM mavenmarket.silver.silver_transactions t
    WHERE t.customer_id IS NOT NULL
),
customer_store_preference AS (
    -- Calculate primary region and store for active customers
    SELECT 
        t.customer_id,
        s.store_id,
        r.region_id,
        r.sales_region AS sales_region_name,
        r.sales_district AS sales_district_name,
        ROW_NUMBER() OVER (
            PARTITION BY t.customer_id 
            ORDER BY COUNT(DISTINCT CONCAT(t.customer_id, '_', t.transaction_date, '_', t.store_id)) DESC, 
                     MAX(t.transaction_date) DESC, 
                     s.store_id ASC
        ) AS preference_rank
    FROM clean_transactions t
    LEFT JOIN mavenmarket.silver.silver_stores s 
        ON t.store_id = CAST(FLOOR(CAST(s.store_id AS DOUBLE)) AS INT)
    LEFT JOIN mavenmarket.silver.silver_regions r 
        ON CAST(FLOOR(CAST(s.region_id AS DOUBLE)) AS INT) = CAST(FLOOR(CAST(r.region_id AS DOUBLE)) AS INT)
    GROUP BY 
        t.customer_id,
        s.store_id,
        r.region_id,
        r.sales_region,
        r.sales_district
),
customer_primary_region AS (
    SELECT 
        customer_id,
        region_id,
        sales_region_name,
        sales_district_name
    FROM customer_store_preference
    WHERE preference_rank = 1
),
customer_sales_summary AS (
    -- Aggregate purchases per customer
    SELECT 
        t.customer_id,
        MIN(t.transaction_date) AS first_purchase_date,
        MAX(t.transaction_date) AS most_recent_purchase_date,
        COUNT(DISTINCT CONCAT(t.customer_id, '_', t.transaction_date, '_', t.store_id)) AS total_orders_count,
        COUNT(DISTINCT t.transaction_date) AS active_purchase_days_count,
        COUNT(DISTINCT t.product_id) AS distinct_products_bought,
        SUM(t.quantity) AS total_units_purchased,
        SUM(t.quantity * COALESCE(p.product_retail_price, 0)) AS customer_lifetime_value,
        SUM(t.quantity * COALESCE(p.product_cost, 0)) AS total_customer_cost,
        SUM(t.quantity * (COALESCE(p.product_retail_price, 0) - COALESCE(p.product_cost, 0))) AS total_profit_generated
    FROM clean_transactions t
    LEFT JOIN mavenmarket.silver.silver_products p 
        ON t.product_id = CAST(FLOOR(CAST(p.product_id AS DOUBLE)) AS INT)
    GROUP BY t.customer_id
),
customer_sales_base AS (
    SELECT 
        -- Complete Master Customer List (Exactly 60,000 Rows)
        c.customer_id,
        c.customer_name,
        c.customer_city,
        c.customer_state,
        c.customer_country,
        c.gender,
        c.marital_status,
        c.yearly_income_amount,
        c.children_at_home_count,
        c.total_children_count,
        c.education_level,
        c.occupation_name,
        c.is_homeowner_flag,

        -- Region Details
        pr.region_id,
        pr.sales_region_name,
        pr.sales_district_name,

        -- Purchase Timelines
        s.first_purchase_date,
        s.most_recent_purchase_date,
        
        -- Aggregated Metrics
        COALESCE(s.total_orders_count, 0) AS total_orders_count,
        COALESCE(s.active_purchase_days_count, 0) AS active_purchase_days_count,
        COALESCE(s.distinct_products_bought, 0) AS distinct_products_bought,
        COALESCE(s.total_units_purchased, 0) AS total_units_purchased,

        -- Financial Metrics
        COALESCE(s.customer_lifetime_value, 0) AS customer_lifetime_value,
        COALESCE(s.total_customer_cost, 0) AS total_customer_cost,
        COALESCE(s.total_profit_generated, 0) AS total_profit_generated

    FROM master_customers c
    LEFT JOIN customer_sales_summary s 
        ON c.customer_id = s.customer_id
    LEFT JOIN customer_primary_region pr 
        ON c.customer_id = pr.customer_id
),
customer_analytics_calculated AS (
    SELECT 
        b.*,

        -- Total Counts
        COUNT(b.customer_id) OVER () AS total_customer_count,
        COUNT(CASE WHEN b.total_orders_count > 0 THEN b.customer_id END) OVER () AS total_active_customers_count,

        -- Financial Ratios
        ROUND(
            b.customer_lifetime_value / NULLIF(b.total_orders_count, 0), 
            2
        ) AS avg_order_value,

        ROUND(
            (b.total_profit_generated / NULLIF(b.customer_lifetime_value, 0)) * 100.0, 
            2
        ) AS customer_profit_margin_pct,

        ROUND(
            b.total_units_purchased / NULLIF(b.total_orders_count, 0), 
            2
        ) AS avg_units_per_order,

        DATEDIFF(CURRENT_DATE(), b.most_recent_purchase_date) AS days_since_last_purchase,

        -- Order Frequency Tiers
        CASE 
            WHEN b.total_orders_count >= 20 THEN 'High Frequency (20+ Orders)'
            WHEN b.total_orders_count BETWEEN 8 AND 19 THEN 'Medium Frequency (8-19 Orders)'
            WHEN b.total_orders_count BETWEEN 1 AND 7 THEN 'Low Frequency (1-7 Orders)'
            ELSE 'No Purchases Yet'
        END AS purchase_frequency_tier,

        -- Portfolio Averages
        ROUND(
            AVG(b.customer_lifetime_value) OVER (), 
            2
        ) AS global_avg_customer_ltv

    FROM customer_sales_base b
)
SELECT 
    c.*,
    DENSE_RANK() OVER (ORDER BY c.customer_lifetime_value DESC) AS ltv_rank_position,
    ROUND(c.customer_lifetime_value - c.global_avg_customer_ltv, 2) AS ltv_variance_from_avg
FROM customer_analytics_calculated c;

### Returns Performance MV

In [0]:
CREATE OR REFRESH MATERIALIZED VIEW mavenmarket.gold.gold_returns_performance
COMMENT "Materialized View: Gold layer aggregated return performance with monthly and all-time sales fallback metrics"
AS
WITH clean_returns AS (
    SELECT 
        CAST(t.return_date AS DATE) AS return_date,
        CAST(FLOOR(CAST(t.store_id AS DOUBLE)) AS INT) AS store_id,
        CAST(FLOOR(CAST(t.product_id AS DOUBLE)) AS INT) AS product_id,
        CAST(t.quantity AS INT) AS return_quantity
    FROM mavenmarket.silver.silver_returns t
    WHERE t.product_id IS NOT NULL AND t.store_id IS NOT NULL
),
clean_transactions AS (
    SELECT 
        CAST(t.transaction_date AS DATE) AS transaction_date,
        CAST(FLOOR(CAST(t.store_id AS DOUBLE)) AS INT) AS store_id,
        CAST(FLOOR(CAST(t.product_id AS DOUBLE)) AS INT) AS product_id,
        CAST(t.quantity AS INT) AS sales_quantity
    FROM mavenmarket.silver.silver_transactions t
    WHERE t.product_id IS NOT NULL AND t.store_id IS NOT NULL
),
returns_base AS (
    -- Aggregate returns directly from silver_returns
    SELECT 
        fr.return_date,
        fr.store_id,
        fr.product_id,
        SUM(fr.return_quantity) AS returned_quantity
    FROM clean_returns fr
    GROUP BY 
        fr.return_date,
        fr.store_id,
        fr.product_id
),
monthly_product_sales AS (
    -- Aggregate sales at Year + Month level for Return Rate % denominator
    SELECT 
        d.year AS year_number,
        d.month AS month_number,
        fs.store_id,
        fs.product_id,
        SUM(fs.sales_quantity) AS monthly_sales_quantity,
        SUM(fs.sales_quantity * p.product_retail_price) AS monthly_sales_revenue
    FROM clean_transactions fs
    JOIN mavenmarket.silver.silver_calendar d 
        ON fs.transaction_date = d.calendar_date
    JOIN mavenmarket.silver.silver_products p 
        ON fs.product_id = p.product_id
    GROUP BY 
        d.year,
        d.month,
        fs.store_id,
        fs.product_id
),
overall_product_sales AS (
    -- All-time product sales fallback for items returned outside their purchase month
    SELECT 
        fs.product_id,
        SUM(fs.sales_quantity) AS total_product_sales_quantity,
        SUM(fs.sales_quantity * p.product_retail_price) AS total_product_sales_revenue
    FROM clean_transactions fs
    JOIN mavenmarket.silver.silver_products p 
        ON fs.product_id = p.product_id
    GROUP BY fs.product_id
)
SELECT 
    -- 1. Date & Trend Attributes
    d.calendar_date AS return_date,
    d.year AS return_year,
    d.quarter AS return_quarter,
    d.month AS return_month_number,
    d.month_name AS return_month_name,
    d.year_month AS return_year_month,
    d.day_name AS return_day_of_week,

    -- 2. Product & Category Attributes
    p.product_id,
    p.product_name,
    p.product_brand AS brand_name,
    p.product_category AS product_category,
    p.product_sku AS sku_code,
    p.product_retail_price AS unit_retail_price,
    p.product_cost AS unit_cost,

    -- 3. Store & Region Attributes
    st.store_id,
    st.store_name,
    st.store_city,
    st.store_state,
    r.sales_region,
    r.sales_district,

    -- 4. Volumetric Measures
    rb.returned_quantity,
    COALESCE(ms.monthly_sales_quantity, 0) AS monthly_gross_sales_quantity,
    COALESCE(ops.total_product_sales_quantity, 0) AS total_product_sales_quantity,

    -- 5. Financial Return Value Calculations
    CAST(rb.returned_quantity * COALESCE(p.product_retail_price, 0.00) AS DECIMAL(18,2)) AS total_return_value_amount,
    CAST(rb.returned_quantity * COALESCE(p.product_cost, 0.00) AS DECIMAL(18,2)) AS total_return_cost_amount,

    -- 6. Fully Null-Safe Key Ratios & KPI Calculations
    -- Return Rate % (Uses Monthly Store Sales -> Fallback to Product Total Sales -> Fallback to 100.00%)
    CASE 
        WHEN COALESCE(ms.monthly_sales_quantity, 0) > 0 THEN 
            ROUND((rb.returned_quantity * 100.0) / ms.monthly_sales_quantity, 2)
        WHEN COALESCE(ops.total_product_sales_quantity, 0) > 0 THEN 
            ROUND((rb.returned_quantity * 100.0) / ops.total_product_sales_quantity, 2)
        ELSE 100.00
    END AS return_rate_pct,

    -- Return Value % (Uses Monthly Store Revenue -> Fallback to Product Total Revenue -> Fallback to 100.00%)
    CASE 
        WHEN COALESCE(ms.monthly_sales_revenue, 0) > 0 THEN 
            ROUND(((rb.returned_quantity * COALESCE(p.product_retail_price, 0.00)) * 100.0) / ms.monthly_sales_revenue, 2)
        WHEN COALESCE(ops.total_product_sales_revenue, 0) > 0 THEN 
            ROUND(((rb.returned_quantity * COALESCE(p.product_retail_price, 0.00)) * 100.0) / ops.total_product_sales_revenue, 2)
        ELSE 100.00
    END AS return_value_pct

FROM returns_base rb

-- Primary Date Join
JOIN mavenmarket.silver.silver_calendar d 
    ON rb.return_date = d.calendar_date

-- Product & Store Joins
JOIN mavenmarket.silver.silver_products p 
    ON rb.product_id = p.product_id

JOIN mavenmarket.silver.silver_stores st 
    ON rb.store_id = st.store_id

JOIN mavenmarket.silver.silver_regions r 
    ON CAST(st.region_id AS INT) = CAST(r.region_id AS INT)

-- Monthly Sales Join
LEFT JOIN monthly_product_sales ms
    ON d.year = ms.year_number
   AND d.month = ms.month_number
   AND rb.store_id = ms.store_id
   AND rb.product_id = ms.product_id

-- Overall Product Sales Fallback Join
LEFT JOIN overall_product_sales ops
    ON rb.product_id = ops.product_id

WHERE rb.returned_quantity > 0;

### Online Orders Performance MV

In [0]:
CREATE OR REFRESH MATERIALIZED VIEW mavenmarket.gold.gold_online_orders_performance
COMMENT "Materialized View: Gold layer aggregated online sales channel metrics, fulfillment performance, customer basket behavior, and return analytics"
AS
WITH clean_online_transactions AS (
    -- Filter strictly for online orders and apply double-to-int cast protections
    SELECT 
        CAST(FLOOR(CAST(t.customer_id AS DOUBLE)) AS INT) AS customer_id,
        CAST(FLOOR(CAST(t.store_id AS DOUBLE)) AS INT) AS store_id,
        CAST(FLOOR(CAST(t.product_id AS DOUBLE)) AS INT) AS product_id,
        CAST(t.transaction_date AS DATE) AS transaction_date,
        CAST(t.quantity AS INT) AS quantity,
        t.transaction_mode
    FROM mavenmarket.silver.silver_transactions t
    WHERE LOWER(TRIM(t.transaction_mode)) = 'online'
      AND t.transaction_date IS NOT NULL
),
online_returns AS (
    -- Identify returned units from online transactions
    SELECT 
        CAST(FLOOR(CAST(r.product_id AS DOUBLE)) AS INT) AS product_id,
        CAST(FLOOR(CAST(r.store_id AS DOUBLE)) AS INT) AS store_id,
        CAST(r.return_date AS DATE) AS return_date,
        SUM(CAST(r.quantity AS INT)) AS total_returned_units
    FROM mavenmarket.silver.silver_returns r
    WHERE r.product_id IS NOT NULL
    GROUP BY ALL
),
base_online_monthly_performance AS (
    SELECT 
        -- Monthly Calendar Attributes
        d.year AS year_number,
        d.quarter AS quarter_number,
        d.month AS month_number,
        d.month_name,
        d.year_month AS year_month_code,

        -- Store & Fulfillment Region
        s.store_id,
        s.store_name,
        s.store_type AS store_type_description,
        s.store_city,
        s.store_state,
        s.store_country,

        -- Region Attributes
        r.region_id,
        r.sales_region AS sales_region_name,
        r.sales_district AS sales_district_name,

        -- Product Attributes
        p.product_id,
        p.product_name,
        p.product_brand AS brand_name,
        p.product_category AS category_name,
        p.product_sku AS sku_code,

        -- Channel Type (Fixed to 'online')
        t.transaction_mode AS channel_type,

        -- Online Aggregated Financial & Volume Metrics
        COUNT(DISTINCT t.customer_id) AS distinct_online_buyers_count,
        COUNT(DISTINCT CONCAT(t.customer_id, '_', t.transaction_date, '_', t.store_id)) AS total_online_orders_count,
        SUM(t.quantity) AS total_online_units_sold,
        COALESCE(SUM(ret.total_returned_units), 0) AS total_online_units_returned,

        SUM(t.quantity * COALESCE(p.product_retail_price, 0)) AS gross_online_revenue_amount,
        SUM(t.quantity * COALESCE(p.product_cost, 0)) AS total_online_cost_amount,
        SUM(t.quantity * (COALESCE(p.product_retail_price, 0) - COALESCE(p.product_cost, 0))) AS gross_online_profit_amount

    FROM clean_online_transactions t
    JOIN mavenmarket.silver.silver_calendar d 
        ON t.transaction_date = d.calendar_date
    LEFT JOIN mavenmarket.silver.silver_stores s 
        ON t.store_id = CAST(FLOOR(CAST(s.store_id AS DOUBLE)) AS INT)
    LEFT JOIN mavenmarket.silver.silver_regions r 
        ON CAST(FLOOR(CAST(s.region_id AS DOUBLE)) AS INT) = CAST(FLOOR(CAST(r.region_id AS DOUBLE)) AS INT)
    LEFT JOIN mavenmarket.silver.silver_products p 
        ON t.product_id = CAST(FLOOR(CAST(p.product_id AS DOUBLE)) AS INT)
    LEFT JOIN online_returns ret 
        ON t.product_id = ret.product_id 
       AND t.store_id = ret.store_id 
       AND t.transaction_date = ret.return_date

    GROUP BY ALL
),
online_with_calculated_kpis AS (
    SELECT 
        *,

        -- Online Financial Ratios
        ROUND(
            (gross_online_profit_amount / NULLIF(gross_online_revenue_amount, 0)) * 100.0, 
            2
        ) AS online_profit_margin_pct,

        -- Average Online Order Value (AOV)
        ROUND(
            gross_online_revenue_amount / NULLIF(total_online_orders_count, 0), 
            2
        ) AS online_average_order_value,

        -- Average Units per Online Basket
        ROUND(
            total_online_units_sold / NULLIF(total_online_orders_count, 0), 
            2
        ) AS avg_units_per_online_order,

        -- Online Return Rate Percentage
        ROUND(
            (total_online_units_returned / NULLIF(total_online_units_sold, 0)) * 100.0, 
            2
        ) AS online_return_rate_pct,

        -- Prior Year Online Revenue (12-month lookback)
        LAG(gross_online_revenue_amount, 12) OVER (
            PARTITION BY store_id, product_id 
            ORDER BY year_number, month_number
        ) AS prior_year_online_revenue,

        -- Prior Year Online Profit
        LAG(gross_online_profit_amount, 12) OVER (
            PARTITION BY store_id, product_id 
            ORDER BY year_number, month_number
        ) AS prior_year_online_profit

    FROM base_online_monthly_performance
)
SELECT 
    *,
    -- Year-over-Year Online Growth Metrics
    CASE 
        WHEN prior_year_online_revenue IS NULL THEN NULL
        ELSE ROUND(gross_online_revenue_amount - prior_year_online_revenue, 2)
    END AS online_revenue_yoy_growth_amount,
    
    ROUND(
        ((gross_online_revenue_amount - prior_year_online_revenue) / NULLIF(prior_year_online_revenue, 0)) * 100.0, 
        2
    ) AS online_revenue_yoy_growth_pct,

    CASE 
        WHEN prior_year_online_profit IS NULL THEN NULL
        ELSE ROUND(gross_online_profit_amount - prior_year_online_profit, 2)
    END AS online_profit_yoy_growth_amount,

    ROUND(
        ((gross_online_profit_amount - prior_year_online_profit) / NULLIF(prior_year_online_profit, 0)) * 100.0, 
        2
    ) AS online_profit_yoy_growth_pct

FROM online_with_calculated_kpis;